# 🛡️ Python Context Managers & The `with` Statement

> **Series:** Learn the Basics | **Focus:** Resource Management, RAII & Clean Architecture

**Context managers** are a powerful construct in Python that allow developers to establish a runtime context for a block of code, automatically allocating resources upon entry and **guaranteeing cleanup upon exit**. 

Most commonly invoked via the `with` statement, context managers eliminate resource leaks (such as unclosed file handles, database connections, threading locks, or network sockets), even when unexpected runtime exceptions occur.

---

## 📋 Table of Contents
1. [Foundations: The Problem & The `with` Statement](#1.-Foundations:-The-Problem-&-The-with-Statement)
2. [The Class-Based Protocol (`__enter__` & `__exit__`)](#2.-The-Class-Based-Protocol)
3. [Exception Handling & Suppression in `__exit__`](#3.-Exception-Handling-&-Suppression-in-__exit__)
4. [Generator-Based Context Managers with `@contextlib.contextmanager`](#4.-Generator-Based-Context-Managers)
5. [Standard Library `contextlib` Power Tools](#5.-Standard-Library-contextlib-Power-Tools)
6. [Asynchronous Context Managers (`async with`)](#6.-Asynchronous-Context-Managers-(async-with))
7. [Real-World Case Studies: Timers, Environment & Transactions](#7.-Real-World-Case-Studies)
8. [Common Pitfalls & Anti-Patterns](#8.-Common-Pitfalls-&-Anti-Patterns)
9. [Hands-On Interactive Challenges](#9.-Hands-On-Interactive-Challenges)
10. [Quick Reference Card & Summary Cheat Sheet](#10.-Quick-Reference-Card-&-Summary-Cheat-Sheet)


---
## 1. Foundations: The Problem & The `with` Statement

### ⚠️ The Problem Without Context Managers
Manually opening and closing resources requires repetitive `try ... finally` boilerplate to avoid leaks when errors occur.


In [ ]:
import tempfile
from pathlib import Path

# 1. The cumbersome manual try...finally pattern
tmp_file = tempfile.NamedTemporaryFile(mode="w+", delete=False)
try:
    tmp_file.write("Manual cleanup required.")
finally:
    tmp_file.close()

# 2. The Pythonic with statement (Context Manager)
# File is guaranteed to close automatically upon exiting the block!
with open(tmp_file.name, "r", encoding="utf-8") as f:
    content = f.read()

print(f"Read content cleanly: '{content}'")
print(f"Is file closed automatically? {f.closed}")


---
## 2. The Class-Based Protocol (`__enter__` & `__exit__`)

Any class can become a context manager by implementing the **Context Management Protocol**:
- `__enter__(self)`: Prepares the resource; its return value is assigned to the `as target` variable.
- `__exit__(self, exc_type, exc_val, exc_tb)`: Executes cleanup unconditionally upon leaving the `with` block.


In [ ]:
class DatabaseConnection:
    def __init__(self, db_name: str):
        self.db_name = db_name
        self.is_connected = False

    def __enter__(self):
        print(f"[ENTER] Connecting to database '{self.db_name}'...")
        self.is_connected = True
        return self

    def query(self, sql: str) -> list[str]:
        if not self.is_connected:
            raise RuntimeError("Cannot query a closed database!")
        return [f"Result row for query: {sql}"]

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"[EXIT] Disconnecting from database '{self.db_name}'...")
        self.is_connected = False
        # Returning None / False lets exceptions propagate normally
        return False

# Usage
with DatabaseConnection("production_users") as db:
    data = db.query("SELECT * FROM users LIMIT 1")
    print(f"  -> {data}")

print(f"State outside with block: is_connected = {db.is_connected}")


---
## 3. Exception Handling & Suppression in `__exit__`

The `__exit__` method receives 3 arguments if an exception occurred inside the `with` block:
- `exc_type`: Exception class (e.g. `ValueError`).
- `exc_val`: Exception instance.
- `exc_tb`: Traceback object.

### 🛡️ Suppression Rule:
- Returning **`True`** suppresses the exception (program continues normally).
- Returning **`False`** (or `None`) allows the exception to propagate.


In [ ]:
class ExceptionIgnorer:
    def __init__(self, target_exception: type[BaseException]):
        self.target_exception = target_exception

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type and issubclass(exc_type, self.target_exception):
            print(f"[SUPPRESSED] Caught and ignored expected error: {exc_type.__name__}: {exc_val}")
            return True  # Suppress exception!
        return False

# Suppressing ZeroDivisionError
with ExceptionIgnorer(ZeroDivisionError):
    print("About to divide by zero...")
    _ = 10 / 0
    print("This line will not execute.")

print("Execution continued safely past the suppressed exception!")


---
## 4. Generator-Based Context Managers with `@contextlib.contextmanager`

Instead of writing a full class with `__enter__` and `__exit__`, `contextlib.contextmanager` turns a generator containing a single `yield` into a context manager:


In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def execution_timer(label: str):
    start_time = time.perf_counter()
    print(f"[{label}] Started...")
    try:
        yield start_time  # Value bound to 'as target'
    finally:
        elapsed = time.perf_counter() - start_time
        print(f"[{label}] Finished in {elapsed * 1000:.3f} ms")

# Usage
with execution_timer("Matrix Calculation"):
    # Simulate processing
    total = sum(x ** 2 for x in range(100_000))


---
## 5. Standard Library `contextlib` Power Tools

Python's `contextlib` module includes battle-tested utilities:
1. `contextlib.suppress(*exceptions)`: Replaces `try...except: pass`.
2. `contextlib.redirect_stdout`: Captures output printed to console into a string buffer.
3. `contextlib.ExitStack`: Programmatically enters and exits a dynamic list of context managers.
4. `contextlib.nullcontext`: No-op context manager for conditional resource management.


In [ ]:
import contextlib
import io

# 1. contextlib.suppress
with contextlib.suppress(KeyError):
    dummy_dict = {"a": 1}
    _ = dummy_dict["missing_key"]
print("contextlib.suppress cleanly handled missing key!")

# 2. contextlib.redirect_stdout
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    print("This message was captured inside the buffer, not printed to terminal!")

captured_text = buffer.getvalue().strip()
print(f"Captured console text: '{captured_text}'")

# 3. contextlib.nullcontext (Optional context)
def run_computation(use_timer: bool = True):
    ctx = execution_timer("Optional Computation") if use_timer else contextlib.nullcontext()
    with ctx:
        return sum(range(50_000))

run_computation(use_timer=False)


---
## 6. Asynchronous Context Managers (`async with`)

For async operations (e.g. database sessions or HTTP clients with `aiohttp` or `httpx`), Python defines `__aenter__` and `__aexit__` with `async with`.


In [ ]:
import asyncio

class AsyncDatabaseSession:
    async def __aenter__(self):
        print("[ASYNC ENTER] Establishing non-blocking socket session...")
        await asyncio.sleep(0.01)  # Simulate non-blocking I/O
        return "Async Session #101"

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("[ASYNC EXIT] Closing non-blocking socket session...")
        await asyncio.sleep(0.01)
        return False

async def main():
    async with AsyncDatabaseSession() as session:
        print(f"  -> Performing async query inside {session}")

await main()


---
## 7. Real-World Case Studies: Timers, Environment & Transactions

### 🔄 Case Study: Atomic Transaction Buffer with Rollback


In [ ]:
class AtomicTransaction:
    """Simulates an in-memory database transaction with rollback on failure."""
    def __init__(self, data_store: dict):
        self.data_store = data_store
        self.backup = {}

    def __enter__(self):
        # Create shallow backup of state
        self.backup = self.data_store.copy()
        return self.data_store

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f"[ROLLBACK] Error occurred ({exc_val}). Restoring original state...")
            self.data_store.clear()
            self.data_store.update(self.backup)
            return True  # Suppress error to demonstrate rollback recovery
        print("[COMMIT] Transaction committed successfully.")
        return False

database = {"account_A": 1000, "account_B": 500}

# Scenario 1: Succeeded transaction
with AtomicTransaction(database) as db:
    db["account_A"] -= 200
    db["account_B"] += 200
print(f"After successful transfer: {database}")

# Scenario 2: Failed transaction with rollback
with AtomicTransaction(database) as db:
    db["account_A"] -= 500
    raise ValueError("Network connection lost mid-transfer!")

print(f"After aborted transfer (Rollback verified): {database}")


---
## 8. Common Pitfalls & Anti-Patterns

### ❌ Pitfall 1: Blanket Exception Suppression
*Anti-Pattern*: Returning `True` unconditionally from `__exit__`, swallowing unexpected bugs like `TypeError` or `NameError`.
*Fix*: Only return `True` for specifically anticipated exception types.

### ❌ Pitfall 2: Omitting `try ... finally` in `@contextmanager`
*Anti-Pattern*:
```python
@contextmanager
def bad_context():
    acquire()
    yield
    release()  # If an exception happens in the with block, release() is NEVER called!
```
*Fix*: Always wrap `yield` inside a `try ... finally: release()` block.


---
## 9. Hands-On Interactive Challenges


In [ ]:
import os
from contextlib import contextmanager

# Challenge 1: Temporary Environment Variable Overrider
@contextmanager
def temporary_env_vars(env_updates: dict[str, str]):
    """Temporarily sets or overrides environment variables and restores them on exit."""
    original_state = {k: os.environ.get(k) for k in env_updates}
    try:
        os.environ.update(env_updates)
        yield
    finally:
        for k, v in original_state.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

# Challenge 2: Atomic List Appender (commits only if no exception occurs)
class AtomicListAppend:
    def __init__(self, target_list: list):
        self.target_list = target_list
        self.buffer = []

    def append(self, item):
        self.buffer.append(item)

    def __enter__(self):
        self.buffer = []
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            self.target_list.extend(self.buffer)
        return False

# Automated verification tests
# Test 1: Temporary environment
test_var = "PYTHON_GYM_TEST_ENV"
assert os.environ.get(test_var) is None

with temporary_env_vars({test_var: "ACTIVE_KEY"}):
    assert os.environ.get(test_var) == "ACTIVE_KEY"

assert os.environ.get(test_var) is None

# Test 2: Atomic List
items = [1, 2, 3]
with AtomicListAppend(items) as appender:
    appender.append(4)
    appender.append(5)

assert items == [1, 2, 3, 4, 5]

# Test 2b: Rollback on exception
try:
    with AtomicListAppend(items) as appender:
        appender.append(6)
        appender.append(7)
        raise RuntimeError("Simulated failure")
except RuntimeError:
    pass

assert items == [1, 2, 3, 4, 5]  # 6 and 7 were discarded!

print("[OK] All Context Manager Challenges Passed!")


---
## 10. Quick Reference Card & Summary Cheat Sheet

### 📊 Context Manager Reference Matrix

| Feature | Syntax | Key Method / Decorator |
| :--- | :--- | :--- |
| **Class Protocol** | `with Class() as obj:` | `__enter__(self)`, `__exit__(self, type, val, tb)` |
| **Generator Protocol**| `@contextmanager def fn():` | `try: yield val finally: cleanup()` |
| **Exception Suppress**| `with suppress(MyError):` | `contextlib.suppress()` |
| **Redirect Output** | `with redirect_stdout(buf):` | `contextlib.redirect_stdout(buffer)` |
| **Dynamic Contexts** | `with ExitStack() as stack:` | `stack.enter_context(ctx)` |
| **Async Context** | `async with AsyncClass():` | `__aenter__(self)`, `__aexit__(self, type, val, tb)` |
